#Distilbert pre trained model 

In [1]:

import pandas as pd 

In [2]:
train_df = pd.read_csv("data/processed/train_20k.csv")
val_df = pd.read_csv("data/processed/validation_10k.csv")
test_df = pd.read_csv("data/processed/test_10k.csv")

In [3]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (20000, 7)
Validation: (10000, 16)
Test: (10000, 16)


In [4]:
print("\nColumns:")
print(train_df.columns.tolist())

print("\nSample:")
display(train_df.head())


Columns:
['id', 'clean_title', 'image_url', 'domain', 'subreddit', 'created_utc', '2_way_label']

Sample:


,id,clean_title,image_url,domain,subreddit,created_utc,2_way_label
0,84nojx,this photo of putin,https://preview.redd.it/ey5rao0mcyl01.jpg?widt...,i.redd.it,photoshopbattles,1.521131e+09,1
1,4x4pit,the name of lenin shines on our victories sovi...,https://external-preview.redd.it/2Mz83vAFfUwLq...,i.imgur.com,propagandaposters,1.470867e+09,0
2,cbvw2jt,i dont think hes the only one alone here,http://i.imgur.com/Q85pUjI.jpg,NaN,psbattle_artwork,1.377556e+09,0
3,dayn4m,this cat in a costume talking to a goku cosplayer,https://preview.redd.it/v88frif6y1g31.jpg?widt...,i.redd.it,subredditsimulator,1.569780e+09,0
4,1ughyb,he survived floor plunge and now walks for cha...,https://external-preview.redd.it/bcm7FU6fOZ9qH...,nypost.com,upliftingnews,1.388928e+09,1


In [5]:
# now check the labels 

print("Train labels:")
print(train_df["2_way_label"].value_counts())

print("\nValidation labels:")
print(val_df["2_way_label"].value_counts())

print("\nTest labels:")
print(test_df["2_way_label"].value_counts())

Train labels:
2_way_label
1    10000
0    10000
Name: count, dtype: int64

Validation labels:
2_way_label
1    5000
0    5000
Name: count, dtype: int64

Test labels:
2_way_label
1    5000
0    5000
Name: count, dtype: int64


In [6]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Transformers: 5.18.0
CUDA: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [7]:
# Select only the columns needed for DistilBERT

train_text = train_df["clean_title"].fillna("").astype(str)
train_labels = train_df["2_way_label"].astype(int)

val_text = val_df["clean_title"].fillna("").astype(str)
val_labels = val_df["2_way_label"].astype(int)

test_text = test_df["clean_title"].fillna("").astype(str)
test_labels = test_df["2_way_label"].astype(int)

print("Training samples:", len(train_text))
print("Validation samples:", len(val_text))
print("Test samples:", len(test_text))

print("\nExample:")
print(train_text.iloc[0])
print("Label:", train_labels.iloc[0])

Training samples: 20000
Validation samples: 10000
Test samples: 10000

Example:
this photo of putin
Label: 1


In [8]:
# select the DIstilbert model (transformer)
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [9]:
#check the Distillbert model 
sample_text = train_text.iloc[0]

tokens = tokenizer(
    sample_text,
    truncation=True,
    max_length=128
)

print("Original text:")
print(sample_text)

print("\nInput IDs:")
print(tokens["input_ids"])

print("\nNumber of tokens:")
print(len(tokens["input_ids"]))

Original text:
this photo of putin

Input IDs:
[101, 2023, 6302, 1997, 22072, 102]

Number of tokens:
6


In [10]:
# tokenize the data 
def tokenize_data(texts):
    return tokenizer(
        texts.tolist(),
        truncation=True,
        padding=True,
        max_length=128
    )

train_encodings = tokenize_data(train_text)
val_encodings = tokenize_data(val_text)
test_encodings = tokenize_data(test_text)

print("Tokenization completed!")

print("Training samples:", len(train_encodings["input_ids"]))
print("Validation samples:", len(val_encodings["input_ids"]))
print("Test samples:", len(test_encodings["input_ids"]))

Tokenization completed!
Training samples: 20000
Validation samples: 10000
Test samples: 10000


# Now create our own dataset using pytorch

In [11]:
import torch
from torch.utils.data import Dataset

class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = FakeNewsDataset(train_encodings, train_labels)
val_dataset = FakeNewsDataset(val_encodings, val_labels)
test_dataset = FakeNewsDataset(test_encodings, test_labels)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 20000
Validation dataset: 10000
Test dataset: 10000


In [12]:
#check the one sample 
sample = train_dataset[0]

print(sample.keys())
print("Input IDs:", sample["input_ids"][:10])
print("Attention Mask:", sample["attention_mask"][:10])
print("Label:", sample["labels"])

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
Input IDs: tensor([  101,  2023,  6302,  1997, 22072,   102,     0,     0,     0,     0])
Attention Mask: tensor([1, 1, 1, 1, 1, 1, 0, 0, 0, 0])
Label: tensor(1)


In [13]:
# Load the DistillBert model

from transformers import AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print("DistilBERT model loaded successfully!")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT model loaded successfully!


# check the GPU 

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "GPU memory reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
GPU memory allocated: 0.0 GB
GPU memory reserved: 0.0 GB


In [15]:
# TrainingArguments is a Hugging Face class that stores all the settings required for training
# Training configuration

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/distilbert_results",

    num_train_epochs=2,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,

    learning_rate=2e-5,
    weight_decay=0.01,

    fp16=True,

    logging_steps=100,

    save_strategy="epoch",
    eval_strategy="epoch",

    load_best_model_at_end=True,

    report_to="none",

    seed=42
)
print("Training configration created Successful")

Training configration created Successful


# now we create the trainer 

In [17]:
from transformers import Trainer
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

print("trainer created successfully")

trainer created successfully


In [18]:
# Test one small batch on GPU

model.eval()

sample_batch = train_dataset[0]

input_ids = sample_batch["input_ids"].unsqueeze(0).to(device)
attention_mask = sample_batch["attention_mask"].unsqueeze(0).to(device)

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

print("GPU test successful!")
print("Logits:", outputs.logits)
print("Predicted class:", torch.argmax(outputs.logits, dim=1).item())

del input_ids, attention_mask, outputs
torch.cuda.empty_cache()

GPU test successful!
Logits: tensor([[-0.0615, -0.0683]], device='cuda:0')
Predicted class: 0


# start Distillbert training 



In [20]:
import time

start_time = time.time()

train_result = trainer.train()

end_time = time.time()

print("Training completed!")
print("Training time:" , round((end_time - start_time ) / 60 , 2) , "minutes")

Epoch,Training Loss,Validation Loss
1,1.103131,0.375761
2,0.778240,0.532708


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed!
Training time: 7.44 minutes


# Test DistilBERT

In [21]:
test_results = trainer.predict(test_dataset)

print("Test Results:")
print(test_results.metrics)

Test Results:
{'test_loss': 0.367978036403656, 'test_runtime': 29.7522, 'test_samples_per_second': 336.11, 'test_steps_per_second': 84.028}


# evalute the model

In [26]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

y_true = test_results.label_ids
y_pred = test_results.predictions.argmax(axis=1)

print("Accuracy_score" , accuracy_score(y_true , y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1 Score:", f1_score(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Accuracy_score 0.8549
Precision: 0.8486932599724897
Recall: 0.8638
F1 Score: 0.856179998017643

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.85      0.85      5000
           1       0.85      0.86      0.86      5000

    accuracy                           0.85     10000
   macro avg       0.86      0.85      0.85     10000
weighted avg       0.86      0.85      0.85     10000


Confusion Matrix:
[[4230  770]
 [ 681 4319]]


# save the result 

In [28]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

y_true = test_results.label_ids
y_pred = test_results.predictions.argmax(axis=1)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.8549
Precision: 0.8486932599724897
Recall: 0.8638
F1 Score: 0.856179998017643


In [29]:
import json 
import os 

os.makedirs("../results" , exist_ok = True)
distilbert_metrics = {
    "model": "DistilBERT",
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "training_samples": 20000,
    "validation_samples": 10000,
    "test_samples": 10000,
    "epochs": 2
}

with open("../results/distilbert_results.json" , "w") as f:
    json.dump(distilbert_metrics , f , indent = 4)

print("Results saved successfully!!")

Results saved successfully!!


In [30]:
model.save_pretrained("../models/distilbert_fake_news")
tokenizer.save_pretrained("../models/distilbert_fake_news")
print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!
